# Inference Pipeline

In [48]:
import os
import json
import torch
import transformers
from transformers import AutoModelForTokenClassification, AutoTokenizer
import sys
import os
import numpy as np
from typing import List, Dict, Set

# Add parent directory to sys.path so we can import from the project root (useful for Jupyter or script)
# os.getcwd()        → returns current working directory, e.g., "/path/to/symptom-ner/v01"
# os.path.join(..., "..") → moves one directory up, i.e., "/path/to/symptom-ner"
# os.path.abspath()  → resolves this to the absolute path
PARENT_DIR = os.path.abspath(os.path.join(os.getcwd(), ".."))
if PARENT_DIR not in sys.path:
    sys.path.insert(0, PARENT_DIR)

from gcp_utils import download_from_gcs, list_bucket_files
from config import settings


# Get the test data and the labels
with open("data/distillbert_splits/test.jsonl", "r") as f:
    test_data = []
    for line in f:
        test_data.append(line)

with open("data/id2label.json", "r") as f:
    id2label = json.load(f)
with open("data/label2id.json", "r") as f:
    label2id = json.load(f)

# Convert id2label keys from strings to integers (JSON loads keys as strings)
if any(isinstance(k, str) for k in id2label.keys()):
    id2label = {int(k): v for k, v in id2label.items()}


# v01/runs/distilbert-base-uncased/run_0/
VERSION = "v01"
MODEL_NAME = "distilbert-base-uncased"  # or "dmis-lab/biobert-base-cased-v1.2" for BioBERT
RUN_IDX = 0  # Change this to test different runs (0, 1, 2, etc.)

GCS_MODEL_PATH = f"{VERSION}/runs/{MODEL_NAME}/run_{RUN_IDX}"
BUCKET_NAME = settings.BUCKET_NAME  # "ner_training_data_results"


In [2]:
# Create a local directory to download the model
LOCAL_MODEL_DIR = f"./downloaded_models/{MODEL_NAME}/run_{RUN_IDX}"

# Download the model directory from GCS
print(f"Downloading model from gs://{BUCKET_NAME}/{GCS_MODEL_PATH}...")
downloaded_path = download_from_gcs(
    gcs_path=GCS_MODEL_PATH,
    local_path=LOCAL_MODEL_DIR,
    bucket_name=BUCKET_NAME
)

# Load the model
model = AutoModelForTokenClassification.from_pretrained(LOCAL_MODEL_DIR)
tokenizer = AutoTokenizer.from_pretrained(LOCAL_MODEL_DIR)

# Move to device
if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"
    
print(f"Using device: {device}")
model.to(device)
model.eval()

✅  Downloaded all files from the folder
✓ Downloaded directory: gs://ner_training_data_results/v01/runs/distilbert-base-uncased/run_0 (24 files) → ./downloaded_models/distilbert-base-uncased/run_0
Using device: mps


DistilBertForTokenClassification(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSdpaAttention(
            (dropout): Dropout(p=0.1, inplace=False)
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=False)
   

In [52]:

def predict_ner(text, model, tokenizer, device="cpu"):
    """
    Predict NER labels for a given text.
    Returns tokenized text and predictions, handling subword tokenization.
    First and last elements of the predictions can be ignored ([CLS] and [SEP])
    """
    # Tokenize with word_ids to properly align subword tokens
    encoding = tokenizer(text, return_tensors="pt", truncation=True)
    # encoding['input_ids'] is of shape [1, seq_len], so we need to get the first row as a list
    input_ids = encoding['input_ids'][0].tolist()
    tokens = tokenizer.convert_ids_to_tokens(input_ids)
    
    # Get word_ids - this groups subword tokens that belong to the same word
    word_ids = encoding.word_ids(batch_index=0)
    
    # Prepare inputs for model 
    inputs = {k: v.to(device) for k, v in encoding.items()}
    
    # Predict
    with torch.no_grad():
        outputs = model(**inputs)
        predictions = torch.argmax(outputs.logits, dim=-1)

    return tokens, predictions

test_text = "Patient reports severe headache and nausea"
tokens, predictions = predict_ner(test_text, model, tokenizer, device=device)
predictions = predictions[0]
print(f"Tokens {len(tokens)}:\n\t{tokens}")
print(f"Predictions {len(predictions)}:\n\t{predictions}")

Tokens 8:
	['[CLS]', 'patient', 'reports', 'severe', 'headache', 'and', 'nausea', '[SEP]']
Predictions 8:
	tensor([4, 4, 4, 1, 3, 3, 3, 4], device='mps:0')


In [53]:
sample = json.loads(test_data[1])
text = sample.get('text')
tokens = sample.get('tokens')
token_label_ids = sample.get('token_label_ids')
print(f"TEXT: {text}")
print(f"TOKENS from test data: {tokens}")
tks, predictions = predict_ner(text, model, tokenizer, device=device)
print(f"Returned tokens: {tks}")
print(f"Predictions: {predictions[0]}")


TEXT: The patient has lymphatic system symptom.
TOKENS from test data: ['[CLS]', 'the', 'patient', 'has', 'l', '##ym', '##pha', '##tic', 'system', 'sy', '##mpt', '##om', '.', '[SEP]']
Returned tokens: ['[CLS]', 'the', 'patient', 'has', 'l', '##ym', '##pha', '##tic', 'system', 'sy', '##mpt', '##om', '.', '[SEP]']
Predictions: tensor([4, 4, 4, 4, 1, 3, 3, 3, 3, 3, 3, 3, 4, 4], device='mps:0')


In [74]:
def predict_word_level(
    text: str,
    model,  # Just leave as model; don't specify transformers.models
    tokenizer,
    id2label: dict,
    device: str,
    SPECIAL_TOKENS:  set = {'[CLS]', '[SEP]'}
):
    encoding = tokenizer(text, return_tensors="pt", truncation=True)
    input_ids = encoding['input_ids'][0].tolist()
    tokens = tokenizer.convert_ids_to_tokens(input_ids)
    word_ids = encoding.word_ids(batch_index=0)
    # To get the actual words that correspond to the word_ids, just split the text
    # (will be correct as long as your tokenizer doesn't do pretokenization with extra rules) - TODO: CHECK THIS
    words = text.split()  # List of actual words from text, mapped by word_ids

    # drop None , special tokens, and predictions in position 0 and -1
    word_ids = [w for w in word_ids if w != None]
    tokens = [t for t in tokens if t not in SPECIAL_TOKENS]

    # send token ids to device
    inputs = {k: v.to(device) for k, v in encoding.items()}
    # Predict
    with torch.no_grad():
        outputs = model(**inputs)
        predictions = torch.argmax(outputs.logits, dim=-1)

    # remove the first and last prediction since they are the predictions for the special tokens
    predictions = predictions[0].cpu().numpy()[1:-1]
    pred_token_labels = [id2label[t] for t in predictions]


    # Map tokens to words and collapse labels
    assert len(word_ids) == len(tokens) and len(tokens) == len(pred_token_labels)
    current_id, past_id = None, None
    word_labels = []
    for idx, word_id in enumerate(word_ids):
        # word_ids will have the same len as word ids
        print(f"IDX: {idx}")
        if idx == 0:
            print("HERe")
            word_labels.append(pred_token_labels[idx])
            continue # skip comparison since it is the first token 

        # Check if current word id is the same as the previous one
        # this will start with index 1 
        if word_id == word_ids[idx-1]:
            # tokens belong to the same word!
            # so they get the same label
            past_token_label = pred_token_labels[idx-1]
            current_token_label = pred_token_labels[idx]

            # IF NOT 'O', the only thing that we must differentiate is the POLARITY of the label
            past_polarity = past_token_label[-3:] if past_token_label != "O" else "O"
            current_polarity = current_token_label[-3:] if current_token_label != "O" else "O"

            if past_polarity == current_polarity:
                word_labels.append(past_token_label[2:])
            else:
                # CONFLICT
                # tokens belong to the same word but tokens were assigned different labels
                word_labels.append(f"CONFLICT-{past_token_label}-{current_token_label}")


    return tokens, pred_token_labels, word_ids, words, word_labels

predict_word_level(text=text, model=model, tokenizer=tokenizer, id2label=id2label, device=device)

IDX: 0
HERe
IDX: 1
IDX: 2
IDX: 3
IDX: 4
IDX: 5
IDX: 6
IDX: 7
IDX: 8
IDX: 9
IDX: 10
IDX: 11


(['the',
  'patient',
  'has',
  'l',
  '##ym',
  '##pha',
  '##tic',
  'system',
  'sy',
  '##mpt',
  '##om',
  '.'],
 ['O',
  'O',
  'O',
  'B-SYMPTOM_POS',
  'I-SYMPTOM_POS',
  'I-SYMPTOM_POS',
  'I-SYMPTOM_POS',
  'I-SYMPTOM_POS',
  'I-SYMPTOM_POS',
  'I-SYMPTOM_POS',
  'I-SYMPTOM_POS',
  'O'],
 [0, 1, 2, 3, 3, 3, 3, 4, 5, 5, 5, 6],
 ['The', 'patient', 'has', 'lymphatic', 'system', 'symptom.'],
 ['O',
  'SYMPTOM_POS',
  'SYMPTOM_POS',
  'SYMPTOM_POS',
  'SYMPTOM_POS',
  'SYMPTOM_POS'])

In [ ]:
def predict_word_level(
    text: str,
    model,  # Just leave as model; don't specify transformers.models
    tokenizer,
    id2label: dict,
    device: str,
    SPECIAL_TOKENS:  set = {'[CLS]', '[SEP]'}
):
    encoding = tokenizer(text, return_tensors="pt", truncation=True)
    input_ids = encoding['input_ids'][0].tolist()
    tokens = tokenizer.convert_ids_to_tokens(input_ids)
    word_ids = encoding.word_ids(batch_index=0)
    # To get the actual words that correspond to the word_ids, just split the text
    # (will be correct as long as your tokenizer doesn't do pretokenization with extra rules) - TODO: CHECK THIS
    words = text.split()  # List of actual words from text, mapped by word_ids

    # drop None , special tokens, and predictions in position 0 and -1
    word_ids = [w for w in word_ids if w != None]
    tokens = [t for t in tokens if t not in SPECIAL_TOKENS]

    # send token ids to device
    inputs = {k: v.to(device) for k, v in encoding.items()}
    # Predict
    with torch.no_grad():
        outputs = model(**inputs)
        predictions = torch.argmax(outputs.logits, dim=-1)

    # remove the first and last prediction since they are the predictions for the special tokens
    predictions = predictions[0].cpu().numpy()[1:-1]
    pred_token_labels = [id2label[t] for t in predictions]


    # Map tokens to words and collapse labels
    assert len(word_ids) == len(tokens) and len(tokens) == len(pred_token_labels)
    current_id, past_id = None, None
    word_labels = ["None"]
    for idx, word_id in enumerate(word_ids):
        # word_ids will have the same len as word ids
        if idx == 0:
            word_labels[idx] = pred_token_labels[idx]
            pass # skip comparison since it is the first token 

        # Check if current word id is the same as the previous one
        # this will start with index 1 
        if word_id == word_ids[idx-1]:
            # tokens belong to the same word!
            # so they get the same label
            past_token_label = pred_token_labels[idx-1]
            current_token_label = pred_token_labels[idx]

            # IF NOT 'O', the only thing that we must differentiate is the POLARITY of the label
            past_polarity = past_token_label[-3:] if past_token_label != "O" else "O"
            current_polarity = current_token_label[-3:] if current_token_label != "O" else "O"

            if past_polarity == current_polarity:
                word_labels[idx]  = past_token_label
            else:
                # CONFLICT
                # tokens belong to the same word but tokens were assigned different labels
                word_labels[idx] = f"CONFLICT-{past_token_label}-{current_token_label}"


    return tokens, pred_token_labels, word_ids, words, word_labels

predict_word_level(text=text, model=model, tokenizer=tokenizer, id2label=id2label, device=device)

IndexError: list assignment index out of range

In [67]:
id2label[3][-3:]

'POS'

In [50]:
text

'The patient has lymphatic system symptom.'

In [20]:
encoding['input_ids']
tokenizer.convert_ids_to_tokens(encoding['input_ids'])

ValueError: only one element tensors can be converted to Python scalars

In [16]:
predictions.cpu().numpy()

array([[4, 4, 4, 1, 3, 3, 3, 4]])

In [ ]:
a = np.array([-100, 4, 4, 0, 2, 2, 2, 4, -100])
b = a[np.where(a != -100)]
b

In [10]:
test_data

['{"text": "Without any tachypnea.", "word_tokens": ["Without", "any", "tachypnea", "."], "word_labels": ["O", "O", "B-SYMPTOM_NEG", "O"], "tokens": ["[CLS]", "without", "any", "ta", "##chy", "##p", "##nea", ".", "[SEP]"], "input_ids": [101, 2302, 2151, 11937, 11714, 2361, 22084, 1012, 102], "token_labels": ["None", "O", "O", "B-SYMPTOM_NEG", "I-SYMPTOM_NEG", "I-SYMPTOM_NEG", "I-SYMPTOM_NEG", "O", "None"], "token_label_ids": [-100, 4, 4, 0, 2, 2, 2, 4, -100]}\n',
 '{"text": "The patient has lymphatic system symptom.", "word_tokens": ["The", "patient", "has", "lymphatic", "system", "symptom", "."], "word_labels": ["O", "O", "O", "B-SYMPTOM_POS", "I-SYMPTOM_POS", "I-SYMPTOM_POS", "O"], "tokens": ["[CLS]", "the", "patient", "has", "l", "##ym", "##pha", "##tic", "system", "sy", "##mpt", "##om", ".", "[SEP]"], "input_ids": [101, 1996, 5776, 2038, 1048, 24335, 21890, 4588, 2291, 25353, 27718, 5358, 1012, 102], "token_labels": ["None", "O", "O", "O", "B-SYMPTOM_POS", "I-SYMPTOM_POS", "I-SYMPT

id2label

In [ ]:
# # Inference function
# def predict_ner(text, model, tokenizer, id2label, device="cpu"):
#     """
#     Predict NER labels for a given text.
#     Returns aligned tokens and labels, handling subword tokenization.
#     """
#     # Tokenize with word_ids to properly align subword tokens
#     encoding = tokenizer(text, return_tensors="pt", truncation=True, return_offsets_mapping=True)
    
#     # Get word_ids - this groups subword tokens that belong to the same word
#     word_ids = encoding.word_ids(batch_index=0)
    
#     # Prepare inputs for model (exclude offset_mapping)
#     inputs = {k: v.to(device) for k, v in encoding.items() if k != "offset_mapping"}
    
#     # Predict
#     with torch.no_grad():
#         outputs = model(**inputs)
#         predictions = torch.argmax(outputs.logits, dim=-1)
    
#     # Get predictions (excluding special tokens)
#     pred_labels = predictions[0].cpu().numpy()
    
#     # Align tokens with labels (group subword tokens by word_id)
#     aligned_tokens = []
#     aligned_labels = []
#     current_word_id = None
#     current_token_parts = []
#     current_label = None
    
#     for i, (word_id, label_id) in enumerate(zip(word_ids, pred_labels)):
#         if word_id is None:  # Skip special tokens [CLS], [SEP], [PAD]
#             continue
        
#         label = id2label[label_id]
#         token_id = encoding["input_ids"][0][i].item()
#         token_text = tokenizer.convert_ids_to_tokens(token_id)
        
#         if word_id != current_word_id:
#             # New word - save previous word if exists
#             if current_word_id is not None and current_token_parts:
#                 aligned_tokens.append("".join(current_token_parts))
#                 aligned_labels.append(current_label)
            
#             # Start new word
#             current_word_id = word_id
#             # Clean token text (remove special prefixes like ## for BERT, Ġ for GPT-style)
#             clean_token = token_text.replace("##", "").replace("Ġ", "").replace("▁", "")
#             current_token_parts = [clean_token]
#             current_label = label
#         else:
#             # Same word - append subword token (remove special prefixes)
#             clean_token = token_text.replace("##", "").replace("Ġ", "").replace("▁", "")
#             current_token_parts.append(clean_token)
#             # Use the label from the first subword token of the word
#             if current_label is None:
#                 current_label = label
    
#     # Don't forget the last word
#     if current_word_id is not None and current_token_parts:
#         aligned_tokens.append("".join(current_token_parts))
#         aligned_labels.append(current_label)
    
#     return aligned_tokens, aligned_labels

# # Test example
# test_text = "Patient reports severe headache and nausea"
# tokens, labels = predict_ner(test_text, model, tokenizer, id2label, device)
# print("Text:", test_text)
# print("Tokens:", tokens)
# print("Labels:", labels)

KeyError: 4